# Financial Market Data Platform: 03. Machine Learning Market Regime Classification
### Zero-Leakage Temporal Modeling, Walk-Forward Validation, and Macroeconomic Regime States

**Author:** Marouane Bizou  
**Role:** Senior Data Scientist / Machine Learning Engineer  
**Focus:** Economic Regime Modeling, Zero Lookahead Bias, HistGradientBoosting, Model Explainability

---
## 1. Problem Formulation & Economic Regimes
Market conditions continuously shift across discrete volatility and momentum states:
- **Regime 0: Low Vol / Bull (Calm Expansion):** Below-average volatility and positive trailing 3-month momentum.
- **Regime 1: Low Vol / Stagnant (Grinding Drift):** Low volatility with stagnant or negative momentum.
- **Regime 2: High Vol / Bear (Crisis / Selloff):** Elevated volatility and negative momentum (market panic).
- **Regime 3: High Vol / Dynamic Rally (Volatile Rebound):** Elevated volatility with strong positive rebound momentum.

### Critical Requirement: Strict Zero Data Leakage
Financial time series violate the i.i.d. assumption. Standard K-Fold cross validation causes catastrophic lookahead leakage by training on future events to predict past states.
Here, we implement **forward-chaining temporal splits**:
- **Train:** $\le 2023-12-31$ (627 trading days)
- **Validation:** $2024-01-01$ to $2024-12-31$ (252 trading days)
- **Out-of-Sample Holdout Test:** $2025-01-01$ to $2025-12-31$ (249 trading days)


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

from src.utils.config import GOLD_DATA_DIR, MODELS_DIR
from src.ml.feature_pipeline import MarketRegimeFeaturePipeline, REGIME_LABELS
from src.ml.predict import MarketRegimePredictor

sns.set_theme(style="whitegrid")
print("ML modules initialized.")


## 2. Feature Pipeline & Regime Target Construction
We extract lagging features (returns, volatilities, moving average ratios, momentum) for benchmark SPY.


In [ ]:
df_gold = pd.read_parquet(GOLD_DATA_DIR / "gold_market_features.parquet")
pipeline = MarketRegimeFeaturePipeline(benchmark_symbol="SPY")
df_ml = pipeline.build_features_and_regimes(df_gold)

train_df, val_df, test_df = pipeline.split_temporal(df_ml)

print(f"Train split:      {len(train_df)} rows ({train_df['date'].min()} to {train_df['date'].max()})")
print(f"Validation split: {len(val_df)} rows ({val_df['date'].min()} to {val_df['date'].max()})")
print(f"Test split:       {len(test_df)} rows ({test_df['date'].min()} to {test_df['date'].max()})")


## 3. Regime Distribution Across Historical Time Periods


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (split_name, data) in zip(axes, [("Train (<=2023)", train_df), ("Val (2024)", val_df), ("Test (2025)", test_df)]):
    counts = data["regime_name"].value_counts()
    sns.barplot(x=counts.values, y=counts.index, ax=ax, palette="viridis")
    ax.set_title(f"{split_name} (N={len(data)})", fontweight="bold")
    ax.set_xlabel("Count")

plt.tight_layout()
plt.show()


## 4. Benchmark Model Evaluation & Test Set Performance
We evaluate the production model (`models/market_regime_model.joblib`) on the unobserved 2025 holdout test set.


In [ ]:
predictor = MarketRegimePredictor()
feature_cols = pipeline.feature_columns

X_test = test_df[feature_cols]
y_test = test_df["regime_target"]

y_pred = predictor.model.predict(X_test)

target_names = [REGIME_LABELS[i] for i in sorted(y_test.unique())]
print("=== OUT-OF-SAMPLE TEST SET CLASSIFICATION REPORT (2025 HOLDOUT) ===")
print(classification_report(y_test, y_pred, target_names=target_names))


## 5. Confusion Matrix Analysis
Examine classification error distribution across market regimes.


In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
            xticklabels=target_names, yticklabels=target_names, ax=ax)
ax.set_title("2025 Holdout Test Set: Confusion Matrix", fontsize=14, fontweight="bold")
ax.set_xlabel("Predicted Regime")
ax.set_ylabel("True Regime")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


## 6. Live Inference & Current Market Regime State
Query the current regime state and probability distribution using `MarketRegimePredictor`.


In [ ]:
latest_pred = predictor.predict_latest_regime(df_gold)

print(f"As of Date:        {latest_pred['as_of_date']}")
print(f"Predicted Regime:  {latest_pred['predicted_regime_name']} (ID: {latest_pred['predicted_regime_id']})")
print("\nRegime Probabilities:")
for regime_name, prob in latest_pred["regime_probabilities"].items():
    print(f"  - {regime_name:<38}: {prob * 100:6.2f}%")

print("\nKey Macro Signals:")
for signal, val in latest_pred["macro_signals"].items():
    print(f"  - {signal:<25}: {val}")
